In [1]:
import OrcFxAPI
from pathlib import Path
import numpy as np

# =========================
# CONFIG
# =========================
owd_dir = Path(r"C:\Users\verav\Desktop\Studie\Afstuderen\Rhino\Draft sensitivity")

# Output mappen (worden aangemaakt als ze nog niet bestaan)
owr_dir  = owd_dir / "OWR_results"
xlsx_dir = owd_dir / "XLSX_results"
owr_dir.mkdir(parents=True, exist_ok=True)
xlsx_dir.mkdir(parents=True, exist_ok=True)

# Omgeving
water_depth = 30.0  # m

# Periodes: stappen van 0.1 s.  <-- pas het bereik hier aan als je smaller wilt
wave_periods = np.arange(1.0, 20.01, 0.1)

# Een golfrichting (genoeg voor toegevoegde massa + radiatiedemping)
wave_headings = [0.0]  # deg

# Dummy traagheidsmomenten: alleen nodig om de validatie te passeren.
# Raakt A / B / excitatie NIET (puur potentiaalstroming), alleen displacement-RAO's.
I_dummy = (3289.0, 3334.0, 5409.0)  # Ixx, Iyy, Izz [t.m2]
mass = [43*10**3]  # kg

In [2]:
# =========================
# HELPER
# =========================
def get_validation(diff):
    """OrcaWave geeft de validatie terug als tuple-achtig object, niet als
    string. Hier platslaan naar tekst zodat .strip()/.count() veilig werkt."""
    def as_text(attr):
        val = getattr(diff, attr, "")
        if not val:
            return ""
        if isinstance(val, str):
            return val
        try:
            return "\n".join(str(x) for x in val)  # tuple / IndexedDataItem
        except TypeError:
            return str(val)
    return (as_text("ValidationInformationText"),
            as_text("ValidationWarningText"),
            as_text("ValidationErrorText"))

In [3]:
# =========================
# BATCH RUN
# =========================
owd_files = [owd_dir / f"Merganser_{i}.owd" for i in range(7, 9)]  # Merganser_1 t/m Merganser_8
print(f"Te verwerken: {len(owd_files)} owd-bestanden")
print(f"Periodes: {len(wave_periods)} (van {wave_periods[0]:.1f} tot {wave_periods[-1]:.1f} s, stap 0.1)")
print(f"Richting(en): {wave_headings}\n")

results = []  # (naam, status)

for owd_file in owd_files:
    print(f"=== {owd_file.name} ===")
    if not owd_file.exists():
        print("  bestand niet gevonden -> overgeslagen\n")
        results.append((owd_file.name, "niet gevonden"))
        continue
    try:
        diff = OrcFxAPI.Diffraction()
        diff.LoadData(str(owd_file))

        # Omgeving
        diff.SetData("WaterDepth", 0, water_depth)

        # Periodes
        diff.SetData("NumberOfPeriodsOrFrequencies", 0, len(wave_periods))
        for i, T in enumerate(wave_periods):
            diff.SetData("PeriodOrFrequency", i, float(T))

        # Richting
        diff.SetData("NumberOfWaveHeadings", 0, len(wave_headings))
        for i, hdg in enumerate(wave_headings):
            diff.SetData("WaveHeading", i, float(hdg))

        # Dummy traagheidstensor -> alleen om validatie ("zero moments of inertia")
        # te passeren. Beinvloedt A/B/excitatie niet, enkel displacement-RAO's.
        diff.SetData("BodyInertiaSpecifiedBy", 0, "Matrix (for a general body)")
        diff.SetData("BodyInertiaTensorOriginType", 0, "Centre of mass")
        diff.SetData("BodyInertiaTensorRx", 0, I_dummy[0])  # Ixx
        diff.SetData("BodyInertiaTensorRy", 1, I_dummy[1])  # Iyy
        diff.SetData("BodyInertiaTensorRz", 2, I_dummy[2])  # Izz
        diff.SetData("BodyMass", 0, mass[0])  # kg

        # Validatie
        info, warnings, errors = get_validation(diff)
        n_warn = warnings.count("\n") + 1 if warnings.strip() else 0
        print(f"  warnings: {n_warn}")
        if errors.strip():
            print("  VALIDATION ERRORS:")
            print("  " + errors.replace("\n", "\n  "))
            print("  -> overgeslagen\n")
            results.append((owd_file.name, "ERROR - overgeslagen"))
            continue

        # Rekenen
        print("  rekenen...")
        diff.Calculate()

        # Opslaan
        owr_path  = owr_dir  / (owd_file.stem + ".owr")
        xlsx_path = xlsx_dir / (owd_file.stem + ".xlsx")
        diff.SaveResults(str(owr_path))
        diff.SaveResultsSpreadsheet(str(xlsx_path))
        print(f"  opgeslagen: {owr_path.name} + {xlsx_path.name}\n")
        results.append((owd_file.name, "OK"))

    except Exception as e:
        print(f"  FOUT: {e}\n")
        results.append((owd_file.name, f"FOUT: {e}"))

# Samenvatting
print("=" * 40)
print("SAMENVATTING")
print("=" * 40)
for naam, status in results:
    print(f"  {naam:25s} {status}")

Te verwerken: 2 owd-bestanden
Periodes: 191 (van 1.0 tot 20.0 s, stap 0.1)
Richting(en): [0.0]

=== Merganser_7.owd ===
  warnings: 3
  rekenen...
  opgeslagen: Merganser_7.owr + Merganser_7.xlsx

=== Merganser_8.owd ===
  warnings: 3
  rekenen...
  opgeslagen: Merganser_8.owr + Merganser_8.xlsx

SAMENVATTING
  Merganser_7.owd           OK
  Merganser_8.owd           OK
